In [11]:
import duckdb
import pandas as pd
import requests
import json
import time
from datetime import date
from typing import Optional

# ── Config ───────────────────────────────────────────────────────────────────

USE_OLLAMA      = False  # Changed to False to avoid Ollama connection issue
OLLAMA_MODEL    = "nomic-embed-text"
OLLAMA_URL      = "http://localhost:11434"
EMBED_DIM       = 384  # Changed from 768 to 384 to match BAAI/bge-small-en-v1.5 output
DB_PATH          = "news_hybrid.duckdb"   # persists to disk — re-run skips embedding

# ── Connect ───────────────────────────────────────────────────────────────────

con = duckdb.connect(DB_PATH)
con.execute("INSTALL vss; LOAD vss;")

# Enable experimental HNSW persistence
con.execute("SET hnsw_enable_experimental_persistence = true;")

print("✓ DuckDB + VSS ready")

# ── Embedding ─────────────────────────────────────────────────────────────────

def embed(text: str) -> list[float]:
    if USE_OLLAMA:
        r = requests.post(f"{OLLAMA_URL}/api/embeddings",
                          json={"model": OLLAMA_MODEL, "prompt": text.lower()},
                          timeout=30)
        r.raise_for_status()
        return r.json()["embedding"]
    else:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer("BAAI/bge-small-en-v1.5")
        return m.encode(text.lower()).tolist()

# ── Schema — metadata-rich, not just text + vector ───────────────────────────
#
# The 2023 design: title, snippet, url, vector
# The 2026 design: add structured metadata columns the SQL layer can filter on
#
# Every metadata column is a WHERE clause waiting to happen.
# The vector is only touched after SQL narrows the candidate set.

# NOTE: Dropping the table to recreate it with the correct embedding dimension.
con.execute("DROP TABLE IF EXISTS articles;")

con.execute(f"""
    CREATE TABLE IF NOT EXISTS articles (
        id              INTEGER PRIMARY KEY,

        -- ── Content ──────────────────────────────────────────────
        title           TEXT NOT NULL,
        snippet         TEXT,
        url             TEXT,

        -- ── Metadata (the SQL layer) ──────────────────────────────
        -- These columns are indexed with standard B-tree indexes.
        -- SQL filters here before a single cosine op runs.
        source          VARCHAR,          -- 'techcrunch', 'wsj', 'reuters'
        category        VARCHAR,          -- 'finance', 'technology', 'hr'
        published_date  DATE,
        -- sentiment       VARCHAR,          -- 'positive', 'negative', 'neutral' -- Removed sentiment
        is_executive    BOOLEAN,          -- mentions C-suite hire/exit?

        -- ── Vector (the similarity layer) ─────────────────────────
        embedding       FLOAT[{EMBED_DIM}]
    )
""")

# Standard B-tree indexes on metadata — these are free and fast
con.execute("CREATE INDEX IF NOT EXISTS idx_source   ON articles(source)")
con.execute("CREATE INDEX IF NOT EXISTS idx_category ON articles(category)")
con.execute("CREATE INDEX IF NOT EXISTS idx_date     ON articles(published_date)")
con.execute("CREATE INDEX IF NOT EXISTS idx_exec     ON articles(is_executive)")

# HNSW on the vector column — for when you want full-corpus search
con.execute(f"""
    CREATE INDEX IF NOT EXISTS idx_hnsw
    ON articles USING HNSW (embedding)
    WITH (metric = 'cosine')
""")
print("✓ Schema + indexes ready (B-tree on metadata, HNSW on embedding)")

# ── Sample data with rich metadata ────────────────────────────────────────────

# NOTE: I've removed `sentiment` from the ARTICLE list as it was inconsistently provided.
ARTICLES = [
    # id, title, snippet, url, source, category, date, is_executive
    (1,  "OpenAI Hires Chief Financial Officer",
         "OpenAI names Sarah Chen as its first CFO amid rapid growth",
         "https://tc.com/1", "techcrunch", "finance", "2023-10-01", True),

    (2,  "Meta Looking for New CFO Amid Restructuring",
         "Meta begins executive search for chief financial officer",
         "https://forbes.com/2", "forbes", "finance", "2023-10-02", True),

    (3,  "Google Names New Chief Technology Officer",
         "Google appoints VP of Engineering to CTO role",
         "https://wsj.com/3", "wsj", "technology", "2023-10-03", True),

    (4,  "Amazon Appoints AWS Chief Technology Officer",
         "AWS names internal leader as its new technology chief",
         "https://cnbc.com/4", "cnbc", "technology", "2023-10-04", True),

    (5,  "Tesla Promotes Internal Candidate to CFO Role",
         "Tesla finance chief steps up after predecessor departs",
         "https://bloomberg.com/5", "bloomberg", "finance", "2023-10-05", True),

    (6,  "AI Startup Raises $200M in Series B Round",
         "Stealth AI company emerges with massive funding",
         "https://nyt.com/6", "nyt", "technology", "2023-10-06", False),

    (7,  "Major Banks Announce Hiring Freeze for 2024",
         "Goldman and JPMorgan halt new headcount amid rate concerns",
         "https://reuters.com/7", "reuters", "finance", "2023-10-07", False),

    (8,  "Climate Tech Company Secures Major Investment",
         "Carbon capture startup closes $150M round led by Breakthrough Energy",
         "https://wired.com/8", "wired", "technology", "2023-10-08", False),

    (9,  "CFO of Fortune 500 Retailer Steps Down Unexpectedly",
         "The departure follows a difficult Q3 earnings report",
         "https://wsj.com/9", "wsj", "finance", "2023-10-09", True),

    (10, "Microsoft Appoints New Chief AI Officer",
         "Microsoft creates new C-suite role to lead AI strategy",
         "https://tc.com/10", "techcrunch", "technology", "2023-10-10", True),
]

# Load only if table is empty (disk persistence means this runs once)
existing = con.execute("SELECT COUNT(*) FROM articles").fetchone()[0]
if existing == 0:
    print(f"\nGenerating embeddings for {len(ARTICLES)} articles...")
    t0 = time.time()
    for row in ARTICLES:
        (id_, title, snippet, url, source, category,
         pub_date, is_exec) = row
        emb = embed(title + " " + (snippet or ""))
        con.execute("""
            INSERT INTO articles (id, title, snippet, url, source, category, published_date, is_executive, embedding) VALUES (?,?,?,?,?,?,?,?,?)
        """, [id_, title, snippet, url, source, category,
              pub_date, is_exec, emb])
    print(f"✓ Embedded and stored in {time.time()-t0:.1f}s")
else:
    print(f"✓ Loaded {existing} articles from disk (embeddings already exist)")

# ── The Core: Hybrid Search ───────────────────────────────────────────────────
#
# This is the 2026 design your 2023 article was pointing toward.
#
# Step 1: SQL WHERE narrows the candidate set using cheap B-tree lookups.
#         If filters are selective, you might go from 10K docs to 50.
#
# Step 2: array_cosine_similarity() runs only on those 50 docs.
#         Exact cosine, not approximate — so results are always correct.
#
# Step 3 (optional): Ollama re-ranks top results with language understanding.
#
# Why this beats pure HNSW:
#   HNSW is approximate. It might return top-100 vectors that fail your filter.
#   Pre-filter guarantees the vector space you search is exactly what you want.

def hybrid_search(
    query: str,
    source: Optional[str]   = None,
    category: Optional[str] = None,
    date_from: Optional[str] = None,
    date_to: Optional[str]   = None,
    exec_only: Optional[bool] = None,
    # sentiment: Optional[str] = None, # Removed sentiment
    top_k: int = 3,
    verbose: bool = True
) -> pd.DataFrame:
    """
    SQL metadata filter → exact cosine on candidate set.

    The SQL layer is free (B-tree). The vector layer only runs
    on the rows that survive the SQL filter.
    """
    # Build WHERE clause from whatever metadata hints were provided
    clauses = []
    if source:      clauses.append(f"source = '{source}'")
    if category:    clauses.append(f"category = '{category}'")
    if date_from:   clauses.append(f"published_date >= '{date_from}'")
    if date_to:     clauses.append(f"published_date <= '{date_to}'")
    if exec_only is not None:
                    clauses.append(f"is_executive = {str(exec_only).upper()}")
    # if sentiment:   clauses.append(f"sentiment = '{sentiment}'") # Removed sentiment

    where_sql = ("WHERE " + " AND ".join(clauses)) if clauses else ""

    # Count the candidate set — shows how much work the SQL layer saved
    candidate_count = con.execute(
        f"SELECT COUNT(*) FROM articles {where_sql}"
    ).fetchone()[0]

    if verbose:
        print(f"  SQL filter: {candidate_count}/{len(ARTICLES)} rows in candidate set")

    if candidate_count == 0:
        return pd.DataFrame(columns=["title", "source", "category", "cosine_sim"])

    # Embed the query once, then run cosine only on the candidate set
    q_emb = embed(query)

    result = con.execute(f"""
        SELECT
            title,
            source,
            category,
            published_date,
            -- sentiment, # Removed sentiment
            is_executive,
            ROUND(
                array_cosine_similarity(
                    embedding::FLOAT[{EMBED_DIM}],
                    {q_emb}::FLOAT[{EMBED_DIM}]
                ), 4
            ) AS cosine_sim
        FROM articles
        {where_sql}
        ORDER BY cosine_sim DESC
        LIMIT {top_k}
    """).df()

    return result




✓ DuckDB + VSS ready
✓ Schema + indexes ready (B-tree on metadata, HNSW on embedding)

Generating embeddings for 10 articles...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Embedded and stored in 26.5s


In [12]:
# ── Showtime ──────────────────────────────────────────────────────────────────

print("***** SHOWTIME - HYBRID SEARCH DEMOS *****")
print("\n" )

# Example 1 — Same query as your 2023 article, no filters (full corpus)
print("═"*65)
print("\n[1] Full corpus search (no metadata filter — 2023 equivalent)")
print("    Query: 'Who is hiring chief financial officer?'")
print("═"*65)
r = hybrid_search("Who is hiring chief financial officer?")
print(r[["title", "source", "cosine_sim"]].to_string(index=False))
print("\n" )

# Example 2 — SQL pre-filter by category, then cosine
print("═"*65)
print("\n[2] Pre-filter: category = 'finance' → cosine on subset")
print("    Query: 'Who is hiring chief financial officer?'")
print("═"*65)
r = hybrid_search("Who is hiring chief financial officer?", category="finance")
print(r[["title", "source", "cosine_sim"]].to_string(index=False))


# Example 3 — Multiple metadata filters
print("\n" + "═"*65)
print("\n[3] Pre-filter: category='finance' AND is_executive=True") # Removed sentiment filter
print("    Query: 'CFO appointment'")
print("═"*65)
r = hybrid_search("CFO appointment",
                  category="finance", exec_only=True) # Removed sentiment filter
print(r[["title", "source", "is_executive", "cosine_sim"]].to_string(index=False)) # Adjusted output columns


# Example 4 — Date-bounded search (e.g. for a specific reporting period)
print("\n" + "═"*65)
print("\n[4] Pre-filter: date range Oct 1–5 (like a reporting window)")
print("    Query: 'technology executive hire'")
print("═"*65)
r = hybrid_search("technology executive hire",
                  date_from="2023-10-01", date_to="2023-10-05")
print(r[["title", "source", "published_date", "cosine_sim"]].to_string(index=False))


# Example 5 — Source-specific (e.g. only trust WSJ for financial news)
print("\n" + "═"*65)
print("\n[5] Pre-filter: source = 'wsj' only")
print("    Query: 'executive departure finance'")
print("═"*65)
r = hybrid_search("executive departure finance", source="wsj")
print(r[["title", "category", "cosine_sim"]].to_string(index=False))


***** SHOWTIME - HYBRID SEARCH DEMOS *****


═════════════════════════════════════════════════════════════════

[1] Full corpus search (no metadata filter — 2023 equivalent)
    Query: 'Who is hiring chief financial officer?'
═════════════════════════════════════════════════════════════════
  SQL filter: 10/10 rows in candidate set


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

                                        title     source  cosine_sim
  Meta Looking for New CFO Amid Restructuring     forbes      0.7794
         OpenAI Hires Chief Financial Officer techcrunch      0.7162
Tesla Promotes Internal Candidate to CFO Role  bloomberg      0.7147


═════════════════════════════════════════════════════════════════

[2] Pre-filter: category = 'finance' → cosine on subset
    Query: 'Who is hiring chief financial officer?'
═════════════════════════════════════════════════════════════════
  SQL filter: 5/10 rows in candidate set


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

                                        title     source  cosine_sim
  Meta Looking for New CFO Amid Restructuring     forbes      0.7794
         OpenAI Hires Chief Financial Officer techcrunch      0.7162
Tesla Promotes Internal Candidate to CFO Role  bloomberg      0.7147

═════════════════════════════════════════════════════════════════

[3] Pre-filter: category='finance' AND is_executive=True
    Query: 'CFO appointment'
═════════════════════════════════════════════════════════════════
  SQL filter: 4/10 rows in candidate set


...

                                        title     source  is_executive  cosine_sim
  Meta Looking for New CFO Amid Restructuring     forbes          True      0.7300
Tesla Promotes Internal Candidate to CFO Role  bloomberg          True      0.6929
         OpenAI Hires Chief Financial Officer techcrunch          True      0.6420

═════════════════════════════════════════════════════════════════

[4] Pre-filter: date range Oct 1–5 (like a reporting window)
    Query: 'technology executive hire'
═════════════════════════════════════════════════════════════════
  SQL filter: 5/10 rows in candidate set


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

                                       title source published_date  cosine_sim
   Google Names New Chief Technology Officer    wsj     2023-10-03      0.6589
Amazon Appoints AWS Chief Technology Officer   cnbc     2023-10-04      0.6181
 Meta Looking for New CFO Amid Restructuring forbes     2023-10-02      0.6139

═════════════════════════════════════════════════════════════════

[5] Pre-filter: source = 'wsj' only
    Query: 'executive departure finance'
═════════════════════════════════════════════════════════════════
  SQL filter: 2/10 rows in candidate set


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

                                              title   category  cosine_sim
CFO of Fortune 500 Retailer Steps Down Unexpectedly    finance      0.6628
          Google Names New Chief Technology Officer technology      0.4905
